<a href="https://colab.research.google.com/github/JuanMejia27/Curso-de-Python-nivel2/blob/main/Proyecto_de_analisis_de_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instalamos librerias

In [1]:
pip install numpy

In [2]:
pip install pandas

In [3]:
pip install matplotlib

In [4]:
pip install seaborn

In [5]:
pip install scipy

In [6]:
pip install statsmodels

Fase 1: Carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

#Configuracion de visualizacion
sns.set(style="whitegrid")

#Carga los datos desde archivos CSV
customers_path = "customers.csv"
logistics_path = "logistics.csv"
products_path = "products.csv"
sales_path = "sales.csv"

#Carga de datos
customers_df = pd.read_csv(customers_path)
logistics_df = pd.read_csv(logistics_path)
products_df = pd.read_csv(products_path)
sales_df = pd.read_csv(sales_path)

#Verifica las primeras filas de cada DataFrame
print(customers_df.head())
print(logistics_df.head())
print(products_df.head())
print(sales_df.head())

Exploracion inicial de datos

In [ ]:
print("\nResumen de clinetes:")
print(customers_df.info())
print(customers_df.describe())

print("\nResumen de logistica:")
print(logistics_df.info())
print(logistics_df.describe())

print("\nResumen de productos:")
print(products_df.info())
print(products_df.describe())

print("\nResumen de ventas:")
print(sales_df.info())
print(sales_df.describe())

Limpieza y preparacion de datos

In [ ]:
# Unificar dormatos de fechas
logistics_df['ShipmentDate'] = pd.to_datetime(logistics_df['ShipmentDate'])
sales_df['Date'] = pd.to_datetime(sales_df['Date'])

#Funcion para revisar valores nulos
def check_missing_values(df, name):
  print(f"Valores nulos en {name}:\n")
  print(df.isnull().sum)

#Recisar valores nulos
check_missing_values(customers_df, "Clientes")
check_missing_values(logistics_df, "Logistica")
check_missing_values(products_df, "Productos")
check_missing_values(sales_df, "Ventas")

#Eliminar duplicados en el DataFrame de ventas
sales_df.drop_duplicates(inplace=True)

Datos limpios

In [ ]:
#Union de ventas con clientes, productos y logistica
sales_combined = sales_df.merge(customers_df, on='CustomerID', how='left')
sales_combined = sales_combined.merge(products_df, on='ProductID', how='left')
sales_combined = sales_combined.merge(logistics_df, left_on='SaleID', right='OrderID', how='left')

#Verificar el tamaño del DataFrame combinado
print(f"Tamaño del DataFrame combinado: {sales_combined.shape}")

#Verificar las primeras filas del DataFrame combinado
print("\nPrimeras filas del DataFrame combinado:")
print(sales_combined.head())

#Crear columnas adicionales
sales_combined['NotRevenue'] = sales_combined['TotalRevenue'] - (sales_combined['Discount']*sales_combined['Quantity'])
sales_combined['DeliveryDelay'] = sales_combined['ActualDeliveryDate'] - sales_combined['EstimatedDelivery']
sales_combined['ProfMargin'] = (sales_combined['NotRevenue'] - sales_combined['UnitPrice'] * sales_combined['Quantity']) / sales_combined['NotRevenue']

#Verificar las primeras filas y verificar que se han añadido correctamente
print("\nPrimeras filas del DataFrame combinado con columnas adicionales:")
print(sales_combined[['NotRevenue', 'DeliveryDelay', 'ProfiMargin']].head())

#Verificar que no existan nulos en las nuevas columnas
print("Valores nulos en las nuevas columnas:")
print(sales_combined[['NotRevenue', 'DeliveryDelay', 'ProfMargin']].isnull().sum())

#Verificar si el calculo de NotRevenue es correcto
print("\nVerificar calculo de NotRevenue:")
print(sales_combined[['TotalRevenue', 'Discount', 'Quantity', 'NotRevenue']].head())

#Verificar si el calculo de DeliveryDelay es correcto
print("\nVerificar calculo de DeliveryDelay:")
print(sales_combined[['ActualDeliveryDate', 'EstimatedDelivery', 'DeliveryDelay']].head())

#Verificar si el calculo de ProfMargin es correcto
print("\nVerificar calculo de ProfMargin:")
print(sales_combined[['NotRevenue', 'UnitPrice', 'Quantity', 'ProfMargin']].head())

#Verificar si existen valores nulos en el DataFrame combinado
print("Valores nulos en el DataFrame combinado:")
print(sales_combined.isnull().sum())

#Visualiza las primeras filas del DataFrame combinado para revisar que todo esta bien
print("Visualiza las primeras filas del DataFrame combinado")
print(sales_combined.head())

import matplotlib.pyplot as plt

#Histograma de NotRevenue, DeliveryDelay y ProfMargin
plt.figure(figsize=(14, 6))
plt.subplot(1, 3, 1)
plt.hist(sales_combined['NotRevenue'], bins=20, color='skyblue', edgecolor='black')
plt.title('Distribucion de NotRevenue')
plt.xlabel('NotRevenue')
plt.ylabel('Frecuencia')

plt.subplot(1, 3, 2)
plt.hist(sales_combined['DeliveryDelay'], bins=20, color='lightcoral', edgecolor='black')
plt.title('Distribucion de DeliveryDelay')
plt.xlabel('DeliveryDelay')
plt.ylabel('Frecuencia')

plt.subplot(1, 3, 3)
plt.hist(sales_combined['ProfMargin'], bins=20, color='lightgreen', edgecolor='black')
plt.title('Distribucion de ProfMargin')
plt.xlabel('ProfMargin')
plt.ylabel('Frecuencia')

plt.tight_layout()
plt.show()

Analisis de datos (EDA)

Distribucion de ventas por categoria

In [ ]:
category_sales = sales_combined.groupby('Category')['NotRevenue'].sum().sort_values(ascending=False)
plt.figure(figsize=(10, 6))
category_sales.plot(kind='bar', color='skyblue')
plt.title('Ventas totales por categoria', fontsize=14)
plt.xlabel('Categoria', fontsize=12)
plt.ylabel('Ingresos netos ($)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.tight_layout()
plt.show()

Impacto de los descuentos en las ventas

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=sales_combined, x='Discount', y='NotRevenue', alpha=0.6, hue='Category', palette='viridis')
plt.title('Impacto de los descuentos en las ventas', fontsize=14)
plt.xlabel('Descuento (%)', fontsize=12)
plt.ylabel('Ingresos netos ($)', fontsize=12)
plt.legend(title('Categoria'))
plt.grid(axis='both', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

Analisis estadistico avanzado

In [ ]:
from scipy import stats
anova_result = stats.f_oneway(
    *[sales_combined[sales_combined['Category'] == cat]['NotRevenue'] for category in sales_combined['Category'].unique()]
)
print("\nResultados del ANOVA: \n")
print(f"Estadistica F: {anova_result.statistic:.2f}, Valor P: {anova_result.pvalue:.4f}")

if anova_result.pvalue < 0.05:
  print("Existe una diferencia significativa en las ventas entre las categorias.")
else:
  print("No existe una diferencia significativa en las ventas entre las categorias.")

Regresion lineal entre ingresos y cantidad comprada

In [ ]:
#Regresion lineal para analizar la relacion entre ingresos y cantidad
x = sales_combined['Quantity']
y = sales_combined['NotRevenue']
x = sm.add_constant(x) #Agregar termino de interpretacion
model = sm.OLS(y, x).fit()
print(model.summary())